# Geldium | Predictive Delinquency Modeling
### Executable model training, honest evaluation, and responsible AI

**Dataset:** `Delinquency_prediction_dataset.xlsx` · **Target:** `Delinquent_Account` (1 = delinquent)  
**Status:** Real models trained and measured on the original dataset; rerun notebook to reproduce.  
**Scope:** Experimental analysis only; not validated for real lending decisions.

> **Important:** The figures below are observed results from the supplied dataset, not GenAI-generated predictions. This notebook is designed to be rerun in the same directory as the Excel file.


## 1. Executive summary — actual experiment

- **Dataset:** 500 customers; 80 delinquent (16.0%).
- **Train / holdout:** 375 / 125 customers, stratified; **holdout positives:** 20.
- **Models:** prior-probability baseline, Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, Neural Network (MLP).
- **Primary model selection metric:** 5-fold training **PR-AUC / average precision**, suitable for the 16% positive class.
- **Selected by cross-validation:** **Logistic Regression**. This is a model selection result, **not evidence of deployment readiness**.
- **Key caution:** the EDA showed very weak individual feature associations. A model cannot be assumed effective unless it beats the baseline reliably on independent data.


## 2. Data import and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                             recall_score, f1_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, classification_report,
                             precision_recall_curve)
from pathlib import Path

DATA_FILE = next((p for p in [Path('data/Delinquency_prediction_dataset.xlsx'), Path('../data/Delinquency_prediction_dataset.xlsx'), Path('Delinquency_prediction_dataset.xlsx')] if p.exists()), Path('data/Delinquency_prediction_dataset.xlsx'))
assert DATA_FILE.exists(), f'Place {DATA_FILE.name} next to this notebook'
df = pd.read_excel(DATA_FILE)
print('Shape:', df.shape)
print('Target counts:\n', df['Delinquent_Account'].value_counts())
display(df.head())

## 3. Leakage-safe feature engineering

Standardize employment labels and create behavioral counts from the six monthly statuses. Drop `Customer_ID` (identifier) and the target from predictors. **Do not use any information recorded after the prediction decision date**; this requires business validation. The original `Missed_Payments` and engineered counts may overlap, so inspect collinearity and ablation results later. Imputation and scaling are **fitted within each training fold**, never on the full dataset.

In [ ]:
def make_features(df):
    data = df.copy()
    data['Employment_Status'] = data['Employment_Status'].replace({
        'EMP':'Employed', 'employed':'Employed', 'retired':'Retired'})
    data['Loan_Balance_Missing'] = data['Loan_Balance'].isna().astype(int)
    months = [f'Month_{i}' for i in range(1,7)]
    for status, col in [('On-time','OnTime_Months_6M'),('Late','Late_Months_6M'),
                        ('Missed','Missed_Months_6M')]:
        data[col] = data[months].eq(status).sum(axis=1)
    data['Payment_Severity_6M'] = data['Late_Months_6M'] + 2*data['Missed_Months_6M']
    return data.drop(columns=['Customer_ID','Delinquent_Account']), data['Delinquent_Account'].astype(int)

X, y = make_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

## 4. Preprocessing and evaluation design

**Numeric:** median imputation → standardization. **Categorical:** most-frequent imputation → one-hot encoding with unseen categories ignored. Use identical preprocessing for all six models to ensure a fair comparison.

**Design:** a single 25% stratified holdout is reserved for final evaluation. The remaining 75% undergoes stratified 5-fold cross-validation for model selection. **No threshold tuning or hyperparameter search is performed on the test set.** Because the sample contains only 80 delinquent cases, holdout results have substantial uncertainty.

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ('numeric', Pipeline([('impute',SimpleImputer(strategy='median')),
                              ('scale',StandardScaler())]), numeric_features),
        ('categorical', Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                                  ('onehot',OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
    ], sparse_threshold=0)

model_specs = {
    'Baseline (prior probability)': DummyClassifier(strategy='prior'),
    'Logistic Regression': LogisticRegression(max_iter=2500, class_weight='balanced', C=1.0, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, min_samples_leaf=15, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=6, min_samples_leaf=5,
                                             class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    'Gradient Boosting': HistGradientBoostingClassifier(max_iter=110, max_leaf_nodes=7,
                                                         l2_regularization=5, learning_rate=0.05, random_state=42),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(20,), alpha=10,
                                          early_stopping=True, max_iter=350, random_state=42)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 5. Model-by-model rationale

| Model | What it learns | Strength | Risk / limitation |
|---|---|---|---|
| Prior-probability baseline | Same delinquency probability for every customer | Essential benchmark | Cannot rank customers |
| **Logistic Regression** | Additive effects on log-odds | Transparent coefficients; simple deployment | May miss nonlinear interactions |
| **Decision Tree** | Human-readable conditional splits | Explains individual decision paths | High variance, easily overfits small samples |
| **Random Forest** | Averaged predictions across many trees | Nonlinear interactions, robustness | Less transparent; may overfit noise |
| **Gradient Boosting** | Sequential improvements to tree-based predictions | Flexible nonlinear learner | Needs careful tuning and calibration |
| **Neural Network (MLP)** | Learned nonlinear transformations | Can learn complex relationships | 500 rows is small; unstable, less interpretable |

**Top five candidate features to investigate (not established predictors):** Credit Utilization, Debt-to-Income Ratio, Credit Score, Missed Payments, Account Tenure. Also test Loan Balance missingness and categorical features. EDA associations were weak, so use cross-validated permutation importance rather than assuming these variables are predictive.

## 6. Train models and calculate real metrics

In [ ]:
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)

rows, fitted_models, test_probabilities = [], {}, {}
for name, estimator in model_specs.items():
    pipe = Pipeline([('preprocessing',make_preprocessor()), ('model',estimator)])
    fold = cross_validate(pipe, X_train, y_train, cv=cv,
                          scoring={'ap':'average_precision','auc':'roc_auc'}, n_jobs=1)
    pipe.fit(X_train,y_train)
    proba = pipe.predict_proba(X_test)[:,1]
    pred = (proba >= 0.50).astype(int)
    cm = confusion_matrix(y_test,pred,labels=[0,1])
    rows.append({'Model':name,
        'CV PR-AUC mean':fold['test_ap'].mean(),
        'CV PR-AUC SD':fold['test_ap'].std(),
        'CV ROC-AUC mean':fold['test_auc'].mean(),
        'Test PR-AUC':average_precision_score(y_test,proba),
        'Test ROC-AUC':roc_auc_score(y_test,proba),
        'Precision @0.5':precision_score(y_test,pred,zero_division=0),
        'Recall @0.5':recall_score(y_test,pred,zero_division=0),
        'F1 @0.5':f1_score(y_test,pred,zero_division=0),
        'Balanced Accuracy':balanced_accuracy_score(y_test,pred),
        'Brier Score':brier_score_loss(y_test,proba),
        'TN':cm[0,0],'FP':cm[0,1],'FN':cm[1,0],'TP':cm[1,1]})
    fitted_models[name] = pipe
    test_probabilities[name] = proba

results = pd.DataFrame(rows).sort_values('CV PR-AUC mean',ascending=False)
display(results.round(3))
results.to_csv('Geldium_Model_Results.csv',index=False)  # save from working directory

## 7. Observed model comparison

The following table was computed from the actual uploaded Excel file using the same split, preprocessing, and model settings. **Cross-validation PR-AUC is the model selection criterion; the test metrics are descriptive holdout evaluation.**

| Model                        |   CV PR-AUC mean |   CV PR-AUC SD |   Test PR-AUC |   Test ROC-AUC |   Recall @0.5 |   Precision @0.5 |
|:-----------------------------|-----------------:|---------------:|--------------:|---------------:|--------------:|-----------------:|
| Baseline (prior probability) |            0.16  |          0     |         0.16  |          0.5   |          0    |            0     |
| Logistic Regression          |            0.208 |          0.037 |         0.213 |          0.457 |          0.25 |            0.102 |
| Decision Tree                |            0.182 |          0.028 |         0.146 |          0.446 |          0.45 |            0.129 |
| Random Forest                |            0.186 |          0.037 |         0.244 |          0.4   |          0    |            0     |
| Gradient Boosting            |            0.177 |          0.036 |         0.186 |          0.411 |          0    |            0     |
| Neural Network (MLP)         |            0.206 |          0.038 |         0.182 |          0.443 |          0    |            0     |

### Reading these metrics

- **PR-AUC / average precision:** primary ranking metric for rare delinquency events; compare with the ~16% prevalence baseline.
- **ROC-AUC:** ranking ability across false-positive rates; 0.5 indicates chance-level discrimination.
- **Recall:** fraction of actual delinquent customers identified at a threshold.
- **Precision:** fraction of flagged customers who were actually delinquent.
- **F1:** balances precision and recall, but ignores true negatives.
- **Brier score:** mean squared probability error; lower is better, but class imbalance and calibration matter.
- **False positives:** may lead to unnecessary intervention; **false negatives:** missed opportunities for support.

A good ranking metric alone does not guarantee calibrated probabilities or fair outcomes.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
order = results.sort_values('CV PR-AUC mean',ascending=False)
ax.bar(order['Model'],order['CV PR-AUC mean'])
ax.axhline(y_train.mean(),color='black',linestyle='--',label='Training prevalence')
ax.set_ylabel('5-fold mean average precision')
ax.set_title('Model selection: cross-validated PR-AUC')
ax.tick_params(axis='x',rotation=30)
ax.legend();plt.tight_layout();plt.show()

## 8. Selected model and risk scoring

**Selected by training-only cross-validation:** **Logistic Regression**. The holdout results above must be interpreted against the prior-probability baseline. If all models perform similarly to the baseline, report **insufficient evidence of useful prediction**, not a successful delinquency classifier.

**Risk score:** model `predict_proba(X)[:,1]`, interpreted as a model-estimated probability **only after calibration validation**. The demonstration risk bands below are illustrative and **not approved business thresholds**. Thresholds should be chosen on validation data using intervention cost, precision/recall, capacity, and fairness checks.

In [ ]:
best_name = results.iloc[0]['Model']
best_model = fitted_models[best_name]
best_prob = test_probabilities[best_name]
scored = pd.DataFrame({
    'Customer_ID': df.loc[X_test.index,'Customer_ID'].to_numpy(),
    'Actual_Delinquency':y_test.to_numpy(),
    'Predicted_Probability':best_prob})
scored['Illustrative_Risk_Band'] = pd.cut(scored['Predicted_Probability'],
    bins=[-0.001,0.15,0.30,0.50,0.70,1.001],
    labels=['Very low','Low','Moderate','High','Very high'])
scored = scored.sort_values('Predicted_Probability',ascending=False)
display(scored.head(10))
scored.to_csv('Geldium_Holdout_Risk_Scores.csv',index=False)
print('Selected:',best_name)

## 9. Threshold analysis and confusion matrices

The default 0.50 threshold is not necessarily appropriate for a 16%-positive task. The following code compares candidate thresholds **for diagnostic illustration only**. Never pick a threshold using test outcomes and then present performance on that same test set as unbiased; tune thresholds inside cross-validation and reserve the holdout for one final evaluation.

In [ ]:
for name in results['Model']:
    p = test_probabilities[name]
    print('\n',name)
    print('Confusion matrix at 0.50:\n',confusion_matrix(y_test,p>=0.5))

# Diagnostic illustration: DO NOT select a deployment threshold on test data.
threshold_demo = []
for t in [0.15,0.25,0.35,0.50,0.65]:
    labels = best_prob>=t
    threshold_demo.append({'Threshold':t,'Flagged':int(labels.sum()),
        'Precision':precision_score(y_test,labels,zero_division=0),
        'Recall':recall_score(y_test,labels,zero_division=0),
        'F1':f1_score(y_test,labels,zero_division=0)})
display(pd.DataFrame(threshold_demo).round(3))

## 10. Explainability and robustness checks

**Logistic Regression:** coefficients describe conditional log-odds in encoded/scaled space; multicollinearity and correlated payment-history features complicate attribution. **Tree ensembles:** use held-out permutation importance (with caution for correlated variables), and optionally SHAP after confirming library compatibility. **MLP:** use model-agnostic permutation importance and sensitivity analysis. Always distinguish *association*, *prediction*, and *causation*.

Check stability with repeated stratified CV, ablation of `Location` and other potential proxies, alternative preprocessing, and confidence intervals via bootstrap. Because data contain only 80 positive outcomes, do not overinterpret feature rankings.

In [ ]:
from sklearn.inspection import permutation_importance
# Model-agnostic permutation importance on holdout (exploratory only).
# Uses average precision, not training-set importance.
perm = permutation_importance(best_model, X_test, y_test, scoring='average_precision',
                              n_repeats=5, random_state=42, n_jobs=1)
importance = pd.DataFrame({'Feature':X_test.columns,
    'Mean_PR_AUC_Drop':perm.importances_mean,
    'Std':perm.importances_std}).sort_values('Mean_PR_AUC_Drop',ascending=False)
display(importance.head(12).round(4))

## 11. Bias, fairness, and responsible AI audit

**Do not assume demographic fairness from the EDA.** The file has no explicit protected-characteristic labels such as race or sex. `Age`, `Location`, `Employment_Status`, and income may be legally sensitive or proxy-related depending on jurisdiction. Compare error rates across **permitted, sufficiently sized groups** and obtain governance/legal review before using any feature in credit decisions.

**Metrics:**

- False-positive rate (FPR) = FP / (FP + TN): non-delinquent customers incorrectly flagged.
- False-negative rate (FNR) = FN / (FN + TP): delinquent customers missed.
- True-positive rate (TPR) = TP / (TP + FN): recall; compare across groups (equal opportunity).
- Equalized odds: compare both TPR and FPR across groups.
- Demographic parity: compare positive prediction rates; not automatically the correct fairness criterion for credit risk.
- Calibration by group: among people assigned similar predicted risk, observed outcomes should be similar.

Group comparisons with very small samples or few positive cases are unstable. Do not assert compliance or absence of discrimination based on these exploratory checks. Predictions should support appropriately reviewed customer-support workflows, **not automatic adverse credit decisions**.

In [ ]:
# Illustrative subgroup audit using employment categories, with small-group caution.
check = X_test[['Employment_Status']].copy()
check['Actual'] = y_test.to_numpy()
check['Predicted'] = (best_prob>=0.5).astype(int)
rows=[]
for group,part in check.groupby('Employment_Status'):
    yt=part['Actual'].to_numpy();yp=part['Predicted'].to_numpy()
    tn,fp,fn,tp=confusion_matrix(yt,yp,labels=[0,1]).ravel()
    rows.append({'Group':group,'n':len(part),'positives':int(yt.sum()),
                 'flag_rate':yp.mean(),'TPR':tp/(tp+fn) if tp+fn else np.nan,
                 'FPR':fp/(fp+tn) if fp+tn else np.nan})
fairness_table=pd.DataFrame(rows)
display(fairness_table.round(3))
print('Exploratory only: these categories are not a substitute for a protected-group fairness audit.')

## 12. What GenAI contributed — and what it did not

GenAI helped scaffold candidate algorithms, leakage-safe preprocessing, metrics, evaluation code, interpretability and fairness checklists, and report structure. **All reported scores are generated by executing scikit-learn on the actual Geldium Excel data.** GenAI-generated code should be inspected and rerun; it cannot certify validity, fairness, causality, or regulatory compliance. Independent validation and expert review remain necessary.

## 13. Final recommendation and next experiment

1. **Judge usefulness relative to baseline.** If cross-validated PR-AUC is close to prevalence, predictive signal is limited.
2. **Prefer a transparent baseline** (Logistic Regression) unless a more complex model demonstrates reproducible, meaningful improvement.
3. **Repeat validation** with confidence intervals and nested or repeated cross-validation before trusting model rankings.
4. **Investigate target provenance and time ordering**; six monthly status variables must precede the delinquency event being predicted.
5. **Use calibrated probabilities** and select operational thresholds only with validated cost/benefit and fairness evidence.
6. **Collect richer, lawful predictive features** if performance remains poor; avoid fabricating signal from a small sample.

**Final distinction:** Training a model is not equivalent to demonstrating a deployable predictor. The purpose of this experiment is to quantify whether this dataset supports useful, reliable delinquency prediction.

# 14. Advanced model visualization dashboard

The following charts are generated from **actual holdout predictions** and training-only cross-validation. No illustrative performance values are used. They provide diagnostic evidence, not proof of deployment readiness. All images are also saved into `geldium_model_figures/`.

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
from sklearn.metrics import PrecisionRecallDisplay
sns.set_theme(style='whitegrid', context='notebook')
FIG_DIR = Path('geldium_model_figures'); FIG_DIR.mkdir(exist_ok=True)
def finish_chart(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{name}.png', dpi=170, bbox_inches='tight')
    plt.show()


## 14.1 Model comparison — cross-validation and holdout

Cross-validation is used for model selection. The holdout scores are shown as an independent check; selecting a model based on the highest holdout score would introduce test-set selection bias.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
rank = results.sort_values('CV PR-AUC mean',ascending=True)
axes[0].barh(rank['Model'],rank['CV PR-AUC mean'],xerr=rank['CV PR-AUC SD'],capsize=3)
axes[0].axvline(y_train.mean(),ls='--',color='gray',label='Training prevalence')
axes[0].set(xlabel='Mean average precision (5-fold CV)',title='Training-only cross-validation')
axes[0].legend()
rank2=results.sort_values('Test PR-AUC',ascending=True)
axes[1].barh(rank2['Model'],rank2['Test PR-AUC'])
axes[1].axvline(y_test.mean(),ls='--',color='gray',label='Test prevalence')
axes[1].set(xlabel='Holdout average precision',title='Unseen holdout test set')
axes[1].legend()
finish_chart('01_pr_auc_comparison')

## 14.2 ROC curves — all models

A random-ranking baseline has ROC-AUC of 0.50. ROC curves summarize the true-positive/false-positive trade-off across thresholds.

In [ ]:
fig, ax=plt.subplots(figsize=(9,6))
for name,prob in test_probabilities.items():
    fpr,tpr,_=roc_curve(y_test,prob)
    auc=roc_auc_score(y_test,prob)
    ax.plot(fpr,tpr,label=f'{name} (AUC {auc:.3f})')
ax.plot([0,1],[0,1],'--',color='gray',label='Random ranking')
ax.set(xlabel='False-positive rate',ylabel='True-positive rate',title='Holdout ROC curves')
ax.legend(fontsize=8,loc='lower right')
finish_chart('02_roc_curves')

## 14.3 Precision–recall curves — all models

Because only 16% of customers are delinquent, PR curves are especially important. The horizontal reference is the holdout delinquency prevalence.

In [ ]:
fig,ax=plt.subplots(figsize=(9,6))
for name,prob in test_probabilities.items():
    precision,recall,_=precision_recall_curve(y_test,prob)
    ap=average_precision_score(y_test,prob)
    ax.plot(recall,precision,label=f'{name} (AP {ap:.3f})')
ax.axhline(y_test.mean(),ls='--',color='gray',label=f'Prevalence {y_test.mean():.1%}')
ax.set(xlabel='Recall',ylabel='Precision',title='Holdout precision–recall curves',ylim=(0,1.03))
ax.legend(fontsize=8)
finish_chart('03_precision_recall_curves')

## 14.4 Confusion matrices at threshold 0.50

Each matrix displays correct and incorrect predictions. The default threshold is illustrative; a business threshold must be selected on validation data using error costs.

In [ ]:
names=list(test_probabilities)
fig,axes=plt.subplots(2,3,figsize=(14,9))
for ax,name in zip(axes.flat,names):
    cm=confusion_matrix(y_test,(test_probabilities[name]>=0.5).astype(int),labels=[0,1])
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax,
                xticklabels=['Not delinquent','Delinquent'],yticklabels=['Not delinquent','Delinquent'])
    ax.set(title=name,xlabel='Predicted',ylabel='Actual')
finish_chart('04_confusion_matrices')

## 14.5 Predicted risk distributions

Compare probability scores for actual delinquent and non-delinquent customers. Heavy overlap signals limited discrimination. Class-weighted models may produce probabilities that require calibration.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for ax,name in zip(axes,['Logistic Regression','Random Forest']):
    prob=test_probabilities[name]
    ax.hist(prob[y_test.to_numpy()==0],bins=15,alpha=.6,density=True,label='Non-delinquent')
    ax.hist(prob[y_test.to_numpy()==1],bins=15,alpha=.6,density=True,label='Delinquent')
    ax.set(xlabel='Predicted delinquency probability',ylabel='Density',title=name)
    ax.legend()
finish_chart('05_risk_score_distributions')

## 14.6 Calibration and reliability

Calibration compares predicted probabilities with observed event frequencies. Curves are exploratory because the holdout contains only 20 positive cases. Lower Brier scores indicate better probability accuracy, but are not enough to establish discrimination.

In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
for name in ['Baseline (prior probability)','Logistic Regression','Random Forest','Gradient Boosting']:
    p=test_probabilities[name]
    obs,pred=calibration_curve(y_test,p,n_bins=5,strategy='quantile')
    ax.plot(pred,obs,marker='o',label=f'{name} (Brier {brier_score_loss(y_test,p):.3f})')
ax.plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
ax.set(xlabel='Mean predicted risk',ylabel='Observed delinquency frequency',title='Holdout calibration (small-sample exploratory)',xlim=(0,1),ylim=(0,1))
ax.legend(fontsize=8)
finish_chart('06_calibration')

## 14.7 Threshold sensitivity for the cross-validation-selected model

These plots illustrate how different thresholds affect precision, recall, and the number of customers flagged. Do not optimize a threshold on the holdout set; choose it through training-only validation and assess once on an untouched test set.

In [ ]:
thresholds=np.linspace(.05,.95,37)
p=best_prob
prec=[precision_score(y_test,p>=t,zero_division=0) for t in thresholds]
rec=[recall_score(y_test,p>=t,zero_division=0) for t in thresholds]
flag=[(p>=t).sum() for t in thresholds]
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].plot(thresholds,prec,marker='.',label='Precision')
axes[0].plot(thresholds,rec,marker='.',label='Recall')
axes[0].set(xlabel='Probability threshold',ylabel='Score',title=f'{best_name}: precision vs recall',ylim=(0,1.05))
axes[0].legend()
axes[1].plot(thresholds,flag,marker='.')
axes[1].set(xlabel='Probability threshold',ylabel='Customers flagged (n)',title='Operational review volume')
finish_chart('07_threshold_sensitivity')

## 14.8 Top-k capture / lift chart

Sort customers by estimated risk and evaluate what fraction of actual delinquent cases is captured in the highest-risk fraction. Lift above 1 would suggest enrichment relative to random targeting. Results are exploratory on a small holdout.

In [ ]:
ranked=pd.DataFrame({'risk':best_prob,'actual':y_test.to_numpy()}).sort_values('risk',ascending=False).reset_index(drop=True)
ranked['population_share']=(np.arange(len(ranked))+1)/len(ranked)
ranked['capture']=ranked['actual'].cumsum()/ranked['actual'].sum()
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(ranked['population_share'],ranked['capture'],label=best_name)
ax.plot([0,1],[0,1],'--',color='gray',label='Random targeting')
ax.set(xlabel='Fraction of customers reviewed (highest risk first)',ylabel='Fraction of delinquencies captured',title='Holdout cumulative gains')
ax.legend()
finish_chart('08_cumulative_gains')

## 14.9 Permutation feature importance

Permutation importance estimates the change in average precision when a feature is shuffled. Negative or near-zero importance is possible and should not be interpreted as evidence of a causal relationship. This is an exploratory holdout diagnostic, not a validated ranking.

In [ ]:
top=importance.head(12).sort_values('Mean_PR_AUC_Drop')
fig,ax=plt.subplots(figsize=(9,6))
ax.barh(top['Feature'],top['Mean_PR_AUC_Drop'],xerr=top['Std'],capsize=3)
ax.axvline(0,color='gray',lw=1)
ax.set(xlabel='Mean change in average precision when permuted',title=f'Exploratory importance: {best_name}')
finish_chart('09_permutation_importance')

## 14.10 Exploratory subgroup audit

Employment status is not a substitute for legally protected-group fairness auditing. Small subgroup sizes can make these rates highly unstable; use this visualization to identify follow-up questions, not to certify fairness.

In [ ]:
audit=fairness_table.set_index('Group')[['flag_rate','TPR','FPR']]
fig,ax=plt.subplots(figsize=(10,5))
audit.plot(kind='bar',ax=ax)
ax.set(xlabel='Employment status',ylabel='Rate',title='Exploratory subgroup flag rate, TPR and FPR',ylim=(0,1))
ax.tick_params(axis='x',rotation=20)
finish_chart('10_exploratory_subgroup_audit')
print('Figures saved in:',FIG_DIR.resolve())

## 15. Final interpretation

The charts must be interpreted together, not selectively. Compare against the 16% prevalence baseline, examine both discrimination and calibration, and do not claim useful customer-level risk prediction unless it generalizes on independent data. **Do not use these exploratory scores for lending decisions.**